<a href="https://colab.research.google.com/github/JasonLou622/Hospital-readmissions-analysis/blob/main/diabetes_readmission_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd, sqlite3 # pandas works with tables
                             # sqlite 3 lets Python talk to a SQL database.
                             # (My question is why sqlite3. I mean I saw more
                             # data analysts use PostgreSQL and MySQL than
                             # SQLite.)

df = pd.read_csv("diabetic_data.csv") # dataframe AS df
# reads the csv file into Python as a table called df.

print(df.shape) # df.shape gives (rows, columns)
df.info()       # info() lists each column with its count of non-empty values
                # and its type.
                # 'object' means text, 'int64' means whole numbers.

(101766, 50)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int

Notes:
1. 'max_glu_serum' has only 5346 values and 'A1Cresult' only 17,018, out of 101,766 rows. Most of those tests are missing.

In [8]:
conn = sqlite3.connect("diabetes.db")
# creates a small database file and opens a connection to it.

df.to_sql("diabetes_data", conn, if_exists="replace", index=False)
# df as a table datebase.
# copies the table from diabetes_data.csv into the database df as a SQL table
# name as diabetes_data
# conn is like a phone line used to talk to the database.
# 3rd one means overwritte
# 'index=False' means don't add an extra row-number column

print(pd.read_sql("SELECT * FROM diabetes_data LIMIT 5", conn))
# pd.read_sql a pandas function
# use and run the context in the double quotes as a snippet of SQL code
# conn, connects to the table in diabetes.db

   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)      ?   
1        149190     55629189        Caucasian  Female  [10-20)      ?   
2         64410     86047875  AfricanAmerican  Female  [20-30)      ?   
3        500364     82442376        Caucasian    Male  [30-40)      ?   
4         16680     42519267        Caucasian    Male  [40-50)      ?   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  ...          No      No                   No

Notes:
1. The column 'weight' shows '?' in the column 'weight'. The dataset uses '?' for missing values.
2. There may be some relationship between gender and age.

In [9]:
# Analysis 1: How many patients were readmitted in under 30 days?
query = """
SELECT readmitted, COUNT(*) AS patients
FROM diabetes_data
GROUP BY readmitted;
"""
print(pd.read_sql(query, conn))


# In the triple quotes, allowing to write a multi-line piece of text.
# Python holds it as a text (doesn’t understand it as SQL.)
# 'pd.read_sql' sends that text through the connection, runs it,
# and brings the answer back as a table as the following.

  readmitted  patients
0        <30     11357
1        >30     35545
2         NO     54864


Notes:
1. Readmissions: 11,357 patients came back within 30 days ('<30', about 11%), 35,545 after 30 days('>30', about 35%), and 54,864 not at all ('NO', about 54%). The three add up to 101,766, so nothing was lost.

In [10]:
# Analysis 2: What is the average time in hospital for each age group
query = '''
SELECT age,
       COUNT(*) AS patients,
       ROUND(AVG(time_in_hospital), 2) AS Average_time_in_hospital
FROM diabetes_data
GROUP BY age;
'''
print(pd.read_sql(query, conn))

        age  patients  Average_time_in_hospital
0    [0-10)       161                      2.55
1   [10-20)       691                      3.19
2   [20-30)      1657                      3.56
3   [30-40)      3775                      3.80
4   [40-50)      9685                      4.04
5   [50-60)     17256                      4.13
6   [60-70)     22483                      4.38
7   [70-80)     26068                      4.59
8   [80-90)     17197                      4.81
9  [90-100)      2793                      4.76


Notes:
1. Found that average hospital stay rises with age, from about 2.5 to 4.8 days.

In [12]:
# Analysis 3: How many patients in each age group were readmitted within 30 days?
query = '''
SELECT age,
       COUNT(readmitted) AS patients_readmitted_30d
FROM diabetes_data
WHERE readmitted = '<30'
GROUP BY age;
'''
print(pd.read_sql(query, conn))

        age  patients_readmitted_30d
0    [0-10)                        3
1   [10-20)                       40
2   [20-30)                      236
3   [30-40)                      424
4   [40-50)                     1027
5   [50-60)                     1668
6   [60-70)                     2502
7   [70-80)                     3069
8   [80-90)                     2078
9  [90-100)                      310


Takeaway Notes:
1. Age group [70-80] has the maximum number of patients readmitted with 30 days, and [0-10] group has the minimum readmission number.

In [15]:
# Analysis 4: Which age group has the most readmissions under 30 days?
query = '''
SELECT
  age,
  COUNT(*) AS patients,
  SUM(1) FILTER (WHERE readmitted = '<30') AS readmitted_30d,
  ROUND(
        (SUM(1) FILTER (WHERE readmitted = '<30')) * 100.0 /
        COUNT(*),
  1) AS pct_readmitted_30d
FROM diabetes_data
GROUP BY age
ORDER BY age;
'''
print(pd.read_sql(query, conn))


        age  patients  readmitted_30d  pct_readmitted_30d
0    [0-10)       161               3                 1.9
1   [10-20)       691              40                 5.8
2   [20-30)      1657             236                14.2
3   [30-40)      3775             424                11.2
4   [40-50)      9685            1027                10.6
5   [50-60)     17256            1668                 9.7
6   [60-70)     22483            2502                11.1
7   [70-80)     26068            3069                11.8
8   [80-90)     17197            2078                12.1
9  [90-100)      2793             310                11.1


Notes:
1. The lowest percentage of readmitted in 30 days yield by age group [0-10], and the highest comes from age group [20-30].
2. The most common percentage of 30-day readmission by age is around 11.1%. Overall, the pct_readmitted_30d peaks is relatively smooth which indicates common expectation percentage of readmission in 30 days by age group is under 15%.

In [26]:
# What about readmission above 30 days?
query = '''
SELECT age,
       COUNT(*) AS patients,
       SUM(1) FILTER (WHERE readmitted = '>30') AS readmitted_over30d,
       ROUND(
             SUM(1) FILTER (WHERE readmitted = '>30') * 100.0 /
             COUNT(*),
       1) AS pct_readmitted_over30d
FROM diabetes_data
GROUP BY age
ORDER BY age;
'''

print(pd.read_sql(query, conn))

        age  patients  readmitted_above30d  pct_readmitted_above30d
0    [0-10)       161                   26                     16.1
1   [10-20)       691                  224                     32.4
2   [20-30)      1657                  510                     30.8
3   [30-40)      3775                 1187                     31.4
4   [40-50)      9685                 3278                     33.8
5   [50-60)     17256                 5917                     34.3
6   [60-70)     22483                 7897                     35.1
7   [70-80)     26068                 9475                     36.3
8   [80-90)     17197                 6223                     36.2
9  [90-100)      2793                  808                     28.9


As expected, the percentage of readmitted in more than 30 days for each group is similar to above (readmission under 30 days). Which suggests only age under 10 years old group patients who are less likely readmitted than other age groups.


In [36]:
# Readmission references for both under and over 30-day windows.
query = '''
SELECT age,
       COUNT(*) AS patients,
       SUM(1) FILTER (WHERE readmitted = '<30' OR readmitted = '>30') AS readmitted_30d,
       ROUND(
             SUM(1) FILTER (WHERE readmitted = '<30' OR readmitted = '>30') * 100.0 /
             COUNT(*),
       1) AS pct_readmitted_30d
FROM diabetes_data
GROUP BY age
ORDER BY age;
'''

print(pd.read_sql(query, conn))

        age  patients  readmitted_30d  pct_readmitted_30d
0    [0-10)       161              29                18.0
1   [10-20)       691             264                38.2
2   [20-30)      1657             746                45.0
3   [30-40)      3775            1611                42.7
4   [40-50)      9685            4305                44.5
5   [50-60)     17256            7585                44.0
6   [60-70)     22483           10399                46.3
7   [70-80)     26068           12544                48.1
8   [80-90)     17197            8301                48.3
9  [90-100)      2793            1118                40.0


Monday Oct 26.
Summary:
1. '101766 entries' means 101766 rows from 0 to 101765; 50 columns from 0 to 49 which align with the result of 'df.shape'.
2. In the resulting table of first cell, the non-null count of columns 22 and 23 are different which could mean some values are missing.
3. Moving to the resulting of second cell, finding the values of the column of "weight" missing or not measured, yet pandas thinks nothing is lost (101,766 non-null count) in the result of the table of the first cell. It likely means the test was not done yet, rather than that someone forgot to record it.
3. 54,864 patients (about 54%) were not readmitted.
5. The dataset refers the percentage of readmission of patients under 10 years old are less likely readmitted than other age groups. Moreover, the total readmission percentage is around 46% which aligns with our expectation.

In [35]:
# Continue:
# Does a longer hospital stay relate to readmission?
query = '''
SELECT
       time_in_hospital,
       ROUND(SUM(1)
             FILTER (WHERE readmitted = '<30') * 100.0 /
             COUNT(*),
       1) AS pct_readmitted_30d
FROM diabetes_data
GROUP BY time_in_hospital

'''

print(pd.read_sql(query, conn))

    time_in_hospital  pct_readmitted_30d
0                  1                 8.2
1                  2                 9.9
2                  3                10.7
3                  4                11.8
4                  5                12.0
5                  6                12.6
6                  7                12.8
7                  8                14.2
8                  9                13.7
9                 10                14.3
10                11                10.5
11                12                13.3
12                13                12.3
13                14                13.0


Notes:
1. This outcome indicates patients who stay in hospital longer are more likely readmitted in 30 days.

In [ ]:
# Clean the data and note its limits
df = df.replace("?", pd.NA)
print(df.isna().sum().sort_values(ascending=False).head(10))


In [ ]:
# Making charts to this dataset
import matplotlib.pyplot as plt
result = pd.read_sql(query, conn)
result.plot(x="age", y="pct_readmitted_30d", kind="bar", legend=False)
plt.ylabel("30-day readmission rate (%)")
plt.title("30-day readmission rate by age group")
plt.tight_layout()
plt.show()